<a href="https://colab.research.google.com/github/Ksenia-Savich/cooperative/blob/main/zamok.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [ ]:
# читаем файл
df = pd.read_csv("/Users/macbookpro/Desktop/универ/питон/замок.test", header=None, names=["text"])

FileNotFoundError: [Errno 2] No such file or directory: '/Users/macbookpro/Desktop/универ/питон/замок.test'

In [ ]:

# размечаем, если за`мок, то это "castle", иначе "lock" (замо`к)
df["label"] = df["text"].apply(lambda x: 1 if "за`мок" in x else 0) #содаём новую колонку label

In [ ]:
# убираем ударение, чтобы иишка не подглядывала
df["text"] = df["text"].str.replace("за`мок", "замок").str.replace("замо`к", "замок")

In [ ]:
# загружаем легкую модель BERT
tokenizer = AutoTokenizer.from_pretrained("cointegrated/rubert-tiny2") #скачивает из интернета и загружает токенизатор для легкой русской модели rubert-tiny2
model = AutoModel.from_pretrained("cointegrated/rubert-tiny2") #загружает саму нейросеть

In [ ]:
# получаем векторы для всех предложений разом
    #прогоняем список предложений, добавляем пустые токены в предл разн длины, обрезаем до макс размера модели, возвращаем в формате PyTorch
inputs = tokenizer(df["text"].tolist(), padding=True, truncation=True, return_tensors="pt")
with torch.no_grad():                         #откл градиенты, нужны только векторы
    #берем вектор [CLS]
    X = model(**inputs).last_hidden_state[:, 0, :].numpy()
y = df["label"].values                        #забирает метки классов 0 и 1 в виде массива

In [ ]:
# обучаем и проверяем
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
clf = LogisticRegression(max_iter=1000).fit(X_train, y_train)
print(f"Точность: {accuracy_score(y_test, clf.predict(X_test)) * 100:.2f}%")

In [ ]:
# сохраняем модель
import pickle
with open("model.pkl", "wb") as f:
    pickle.dump(clf, f)